# Prepare PhysioNet A+B Train / C Test Long Time-Series Data

This notebook prepares leakage-safe long-format time-series files for the ICU mortality project.

## Goal

The time-series dataset is split using the same logic as the tabular preprocessing notebook:

- **Set A + Set B** are combined as the training time-series split.
- **Set C** is kept as the held-out test time-series split.
- The notebook is designed to be run from the `code/` folder.
- The input CSV files should be in the sibling `data/` folder.
- The output CSV files will be saved in the sibling `output/` folder.

## Important notes

This notebook does **not** use Set C to learn preprocessing decisions. It only assigns Set C to the test split.

For time-series modeling, rows where `Parameter` is missing cannot be used as valid variables. By default, this notebook drops rows with missing `Parameter`.

The original long-format files include rows where `Parameter == "RecordID"`. Those rows are not physiological measurements and can create an identifier-like feature if the data are later pivoted. By default, this notebook drops those rows while keeping the `RecordID` column as the patient identifier.


In [1]:
# ============================================================
# 0. Imports
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)


In [2]:
# ============================================================
# 1. Path setup
# ------------------------------------------------------------
# Expected project structure:
#
# project_root/
#   code/      <- run this notebook here
#   data/      <- input CSV files here
#   output/    <- output files will be saved here
# ============================================================

CODE_DIR = Path.cwd()
PROJECT_ROOT = CODE_DIR.parent if CODE_DIR.name.lower() == "code" else CODE_DIR
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"CODE_DIR     = {CODE_DIR}")
print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"DATA_DIR     = {DATA_DIR}")
print(f"OUTPUT_DIR   = {OUTPUT_DIR}")


CODE_DIR     = c:\Users\junse\Documents\research\IUBDC 2026\code
PROJECT_ROOT = c:\Users\junse\Documents\research\IUBDC 2026
DATA_DIR     = c:\Users\junse\Documents\research\IUBDC 2026\data
OUTPUT_DIR   = c:\Users\junse\Documents\research\IUBDC 2026\output


In [3]:
# ============================================================
# 2. Locate and load long time-series files
# ============================================================

def find_required_file(data_dir, candidates):
    """Return the first existing file from a list of candidate file names."""
    for name in candidates:
        path = data_dir / name
        if path.exists():
            return path
    raise FileNotFoundError(
        "Could not find any of these required files in the data folder:\n"
        + "\n".join(str(data_dir / name) for name in candidates)
    )

# Main expected names. Extra candidates are included only to make the notebook robust
# if a browser/download renamed the files with suffixes like (1) or (2).
SET_A_TS_FILE = find_required_file(DATA_DIR, [
    "set_a_long_timeseries.csv",
    "set_a_long_timeseries(1).csv",
])
SET_B_TS_FILE = find_required_file(DATA_DIR, [
    "set_b_long_timeseries.csv",
    "set_b_long_timeseries(1).csv",
    "set_b_long_timeseries(2).csv",
])
SET_C_TS_FILE = find_required_file(DATA_DIR, [
    "set_c_long_timeseries.csv",
    "set_c_long_timeseries(1).csv",
])

print("Using files:")
print("Set A:", SET_A_TS_FILE)
print("Set B:", SET_B_TS_FILE)
print("Set C:", SET_C_TS_FILE)

set_a_ts = pd.read_csv(SET_A_TS_FILE, low_memory=False)
set_b_ts = pd.read_csv(SET_B_TS_FILE, low_memory=False)
set_c_ts = pd.read_csv(SET_C_TS_FILE, low_memory=False)

print("\nLoaded shapes:")
print("Set A long time-series:", set_a_ts.shape)
print("Set B long time-series:", set_b_ts.shape)
print("Set C long time-series:", set_c_ts.shape)


Using files:
Set A: c:\Users\junse\Documents\research\IUBDC 2026\data\set_a_long_timeseries.csv
Set B: c:\Users\junse\Documents\research\IUBDC 2026\data\set_b_long_timeseries.csv
Set C: c:\Users\junse\Documents\research\IUBDC 2026\data\set_c_long_timeseries.csv

Loaded shapes:
Set A long time-series: (1757980, 7)
Set B long time-series: (1762535, 7)
Set C long time-series: (1765303, 7)


In [4]:
# ============================================================
# 3. Validate required columns and basic structure
# ============================================================

REQUIRED_COLUMNS = [
    "RecordID",
    "source_file",
    "Time",
    "time_minutes",
    "Parameter",
    "Value",
    "Value_numeric",
]

for split_name, df in [("Set A", set_a_ts), ("Set B", set_b_ts), ("Set C", set_c_ts)]:
    missing_cols = [c for c in REQUIRED_COLUMNS if c not in df.columns]
    if missing_cols:
        raise ValueError(f"{split_name} is missing required columns: {missing_cols}")

summary_before = []

for split_name, df in [("A", set_a_ts), ("B", set_b_ts), ("C", set_c_ts)]:
    summary_before.append({
        "set": split_name,
        "n_rows": len(df),
        "n_unique_patients": df["RecordID"].nunique(),
        "n_parameters_non_missing": df["Parameter"].nunique(dropna=True),
        "missing_parameter_rows": int(df["Parameter"].isna().sum()),
        "missing_value_numeric_rows": int(df["Value_numeric"].isna().sum()),
        "min_time_minutes": df["time_minutes"].min(),
        "max_time_minutes": df["time_minutes"].max(),
    })

summary_before_df = pd.DataFrame(summary_before)
display(summary_before_df)

print("Parameters in Set A:")
print(sorted(set_a_ts["Parameter"].dropna().unique()))


,set,n_rows,n_unique_patients,n_parameters_non_missing,missing_parameter_rows,missing_value_numeric_rows,min_time_minutes,max_time_minutes
0,A,1757980,4000,42,0,0,0,2880
1,B,1762535,4000,42,0,0,0,2880
2,C,1765303,4000,42,2403,0,0,2880


Parameters in Set A:
['ALP', 'ALT', 'AST', 'Age', 'Albumin', 'BUN', 'Bilirubin', 'Cholesterol', 'Creatinine', 'DiasABP', 'FiO2', 'GCS', 'Gender', 'Glucose', 'HCO3', 'HCT', 'HR', 'Height', 'ICUType', 'K', 'Lactate', 'MAP', 'MechVent', 'Mg', 'NIDiasABP', 'NIMAP', 'NISysABP', 'Na', 'PaCO2', 'PaO2', 'Platelets', 'RecordID', 'RespRate', 'SaO2', 'SysABP', 'Temp', 'TroponinI', 'TroponinT', 'Urine', 'WBC', 'Weight', 'pH']


In [5]:
# ============================================================
# 4. Cleaning rules for long-format modeling data
# ------------------------------------------------------------
# These rules do not learn anything from Set C. They only remove
# unusable or identifier-like rows from each split.
# ============================================================

DROP_ROWS_WITH_MISSING_PARAMETER = True

# Parameter == "RecordID" is redundant with the RecordID column.
# If kept and later pivoted, it can become an identifier-like feature.
DROP_RECORDID_PARAMETER_ROWS = True

def clean_long_timeseries(df, set_id, split):
    cleaned = df.copy()

    # Add explicit split labels for later tracking.
    cleaned["set_id"] = set_id
    cleaned["split"] = split

    # Standardize numeric columns.
    cleaned["RecordID"] = pd.to_numeric(cleaned["RecordID"], errors="coerce").astype("Int64")
    cleaned["time_minutes"] = pd.to_numeric(cleaned["time_minutes"], errors="coerce")
    cleaned["Value_numeric"] = pd.to_numeric(cleaned["Value_numeric"], errors="coerce")

    # Keep a summary of dropped rows.
    n_start = len(cleaned)
    n_missing_parameter = int(cleaned["Parameter"].isna().sum())
    n_recordid_parameter = int((cleaned["Parameter"] == "RecordID").sum())

    if DROP_ROWS_WITH_MISSING_PARAMETER:
        cleaned = cleaned[cleaned["Parameter"].notna()].copy()

    if DROP_RECORDID_PARAMETER_ROWS:
        cleaned = cleaned[cleaned["Parameter"] != "RecordID"].copy()

    dropped_summary = {
        "set_id": set_id,
        "split": split,
        "n_rows_before": n_start,
        "missing_parameter_rows_before": n_missing_parameter,
        "recordid_parameter_rows_before": n_recordid_parameter,
        "n_rows_after": len(cleaned),
        "n_rows_dropped_total": n_start - len(cleaned),
    }

    return cleaned, dropped_summary

set_a_clean, drop_a = clean_long_timeseries(set_a_ts, set_id="A", split="train")
set_b_clean, drop_b = clean_long_timeseries(set_b_ts, set_id="B", split="train")
set_c_clean, drop_c = clean_long_timeseries(set_c_ts, set_id="C", split="test")

drop_summary_df = pd.DataFrame([drop_a, drop_b, drop_c])
display(drop_summary_df)


,set_id,split,n_rows_before,missing_parameter_rows_before,recordid_parameter_rows_before,n_rows_after,n_rows_dropped_total
0,A,train,1757980,0,4000,1753980,4000
1,B,train,1762535,0,4000,1758535,4000
2,C,test,1765303,2403,4000,1758900,6403


In [6]:
# ============================================================
# 5. Create A+B training time-series and C test time-series
# ============================================================

time_series_train = pd.concat([set_a_clean, set_b_clean], axis=0, ignore_index=True)
time_series_test = set_c_clean.copy()

# Sort for reproducibility and easier debugging.
sort_cols = ["RecordID", "time_minutes", "Parameter"]
time_series_train = time_series_train.sort_values(sort_cols, kind="mergesort").reset_index(drop=True)
time_series_test = time_series_test.sort_values(sort_cols, kind="mergesort").reset_index(drop=True)

print("time_series_train shape:", time_series_train.shape)
print("time_series_test shape: ", time_series_test.shape)

assert set(time_series_train["split"].unique()) == {"train"}
assert set(time_series_test["split"].unique()) == {"test"}

# Make sure patient IDs do not overlap between train and test.
train_ids = set(time_series_train["RecordID"].dropna().astype(int).unique())
test_ids = set(time_series_test["RecordID"].dropna().astype(int).unique())
overlap_ids = train_ids.intersection(test_ids)

print("Train unique patients:", len(train_ids))
print("Test unique patients: ", len(test_ids))
print("Overlapping patient IDs:", len(overlap_ids))

if overlap_ids:
    raise ValueError("Train and test RecordID overlap detected. This should not happen.")


time_series_train shape: (3512515, 9)
time_series_test shape:  (1758900, 9)
Train unique patients: 8000
Test unique patients:  4000
Overlapping patient IDs: 0


In [7]:
# ============================================================
# 6. Optional: save matching y files if outcome CSVs are available
# ------------------------------------------------------------
# This is useful if you later train a time-series model and need
# RecordID-level labels aligned with the long-format data.
#
# If these files are not available, this cell will skip label export.
# ============================================================

TARGET = "in_hospital_death"

OUTCOME_A_FILE = DATA_DIR / "set_a_ml_ready_with_outcomes.csv"
OUTCOME_B_FILE = DATA_DIR / "set_b_ml_ready_with_outcomes.csv"
OUTCOME_C_FILE = DATA_DIR / "set_c_ml_ready_with_outcomes.csv"

LABELS_AVAILABLE = all(p.exists() for p in [OUTCOME_A_FILE, OUTCOME_B_FILE, OUTCOME_C_FILE])

if LABELS_AVAILABLE:
    outcome_a = pd.read_csv(OUTCOME_A_FILE, usecols=["RecordID", TARGET])
    outcome_b = pd.read_csv(OUTCOME_B_FILE, usecols=["RecordID", TARGET])
    outcome_c = pd.read_csv(OUTCOME_C_FILE, usecols=["RecordID", TARGET])

    time_series_y_train = pd.concat([outcome_a, outcome_b], axis=0, ignore_index=True)
    time_series_y_test = outcome_c.copy()

    time_series_y_train["split"] = "train"
    time_series_y_test["split"] = "test"

    time_series_y_train = time_series_y_train.sort_values("RecordID").reset_index(drop=True)
    time_series_y_test = time_series_y_test.sort_values("RecordID").reset_index(drop=True)

    # Sanity checks: all time-series patients should have labels.
    label_train_ids = set(time_series_y_train["RecordID"].astype(int).unique())
    label_test_ids = set(time_series_y_test["RecordID"].astype(int).unique())

    missing_train_labels = train_ids - label_train_ids
    missing_test_labels = test_ids - label_test_ids

    if missing_train_labels:
        raise ValueError(f"Missing labels for {len(missing_train_labels)} train patients.")
    if missing_test_labels:
        raise ValueError(f"Missing labels for {len(missing_test_labels)} test patients.")

    print("Label files created.")
    print("time_series_y_train shape:", time_series_y_train.shape)
    print("time_series_y_test shape: ", time_series_y_test.shape)
    print("Train death rate:", time_series_y_train[TARGET].mean())
    print("Test death rate: ", time_series_y_test[TARGET].mean())
else:
    time_series_y_train = None
    time_series_y_test = None
    print("Outcome files were not found, so label export was skipped.")
    print("Expected files:")
    print(OUTCOME_A_FILE)
    print(OUTCOME_B_FILE)
    print(OUTCOME_C_FILE)


Label files created.
time_series_y_train shape: (8000, 3)
time_series_y_test shape:  (4000, 3)
Train death rate: 0.14025
Test death rate:  0.14625


In [8]:
# ============================================================
# 7. Save outputs
# ============================================================

TAG = "a_plus_b_train_c_test"

TIME_SERIES_TRAIN_OUT = OUTPUT_DIR / f"time_series_train_{TAG}.csv"
TIME_SERIES_TEST_OUT = OUTPUT_DIR / f"time_series_test_{TAG}.csv"

TIME_SERIES_SUMMARY_OUT = OUTPUT_DIR / f"time_series_split_summary_{TAG}.csv"
TIME_SERIES_DROP_SUMMARY_OUT = OUTPUT_DIR / f"time_series_dropped_rows_summary_{TAG}.csv"
TIME_SERIES_METADATA_OUT = OUTPUT_DIR / f"time_series_metadata_{TAG}.json"

time_series_train.to_csv(TIME_SERIES_TRAIN_OUT, index=False)
time_series_test.to_csv(TIME_SERIES_TEST_OUT, index=False)

# Save labels only if outcome files were available.
if LABELS_AVAILABLE:
    TIME_SERIES_Y_TRAIN_OUT = OUTPUT_DIR / f"time_series_y_train_{TAG}.csv"
    TIME_SERIES_Y_TEST_OUT = OUTPUT_DIR / f"time_series_y_test_{TAG}.csv"

    time_series_y_train.to_csv(TIME_SERIES_Y_TRAIN_OUT, index=False)
    time_series_y_test.to_csv(TIME_SERIES_Y_TEST_OUT, index=False)
else:
    TIME_SERIES_Y_TRAIN_OUT = None
    TIME_SERIES_Y_TEST_OUT = None

summary_after = pd.DataFrame([
    {
        "split": "train_A_plus_B",
        "n_rows": len(time_series_train),
        "n_unique_patients": time_series_train["RecordID"].nunique(),
        "n_parameters": time_series_train["Parameter"].nunique(dropna=True),
        "min_time_minutes": time_series_train["time_minutes"].min(),
        "max_time_minutes": time_series_train["time_minutes"].max(),
        "missing_parameter_rows": int(time_series_train["Parameter"].isna().sum()),
        "missing_value_numeric_rows": int(time_series_train["Value_numeric"].isna().sum()),
    },
    {
        "split": "test_C",
        "n_rows": len(time_series_test),
        "n_unique_patients": time_series_test["RecordID"].nunique(),
        "n_parameters": time_series_test["Parameter"].nunique(dropna=True),
        "min_time_minutes": time_series_test["time_minutes"].min(),
        "max_time_minutes": time_series_test["time_minutes"].max(),
        "missing_parameter_rows": int(time_series_test["Parameter"].isna().sum()),
        "missing_value_numeric_rows": int(time_series_test["Value_numeric"].isna().sum()),
    },
])

summary_after.to_csv(TIME_SERIES_SUMMARY_OUT, index=False)
drop_summary_df.to_csv(TIME_SERIES_DROP_SUMMARY_OUT, index=False)

metadata = {
    "tag": TAG,
    "input_files": {
        "set_a_time_series": str(SET_A_TS_FILE),
        "set_b_time_series": str(SET_B_TS_FILE),
        "set_c_time_series": str(SET_C_TS_FILE),
    },
    "output_files": {
        "time_series_train": str(TIME_SERIES_TRAIN_OUT),
        "time_series_test": str(TIME_SERIES_TEST_OUT),
        "time_series_y_train": str(TIME_SERIES_Y_TRAIN_OUT) if TIME_SERIES_Y_TRAIN_OUT else None,
        "time_series_y_test": str(TIME_SERIES_Y_TEST_OUT) if TIME_SERIES_Y_TEST_OUT else None,
        "summary": str(TIME_SERIES_SUMMARY_OUT),
        "dropped_rows_summary": str(TIME_SERIES_DROP_SUMMARY_OUT),
    },
    "cleaning_rules": {
        "drop_rows_with_missing_parameter": DROP_ROWS_WITH_MISSING_PARAMETER,
        "drop_recordid_parameter_rows": DROP_RECORDID_PARAMETER_ROWS,
    },
    "notes": [
        "Set A and Set B are combined as the training time-series split.",
        "Set C is kept as the held-out test time-series split.",
        "No preprocessing decision is learned from Set C.",
        "The RecordID column is kept as an identifier, but rows where Parameter == RecordID are dropped by default.",
    ],
}

with open(TIME_SERIES_METADATA_OUT, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print("Saved files:")
print(TIME_SERIES_TRAIN_OUT)
print(TIME_SERIES_TEST_OUT)
if LABELS_AVAILABLE:
    print(TIME_SERIES_Y_TRAIN_OUT)
    print(TIME_SERIES_Y_TEST_OUT)
print(TIME_SERIES_SUMMARY_OUT)
print(TIME_SERIES_DROP_SUMMARY_OUT)
print(TIME_SERIES_METADATA_OUT)


Saved files:
c:\Users\junse\Documents\research\IUBDC 2026\output\time_series_train_a_plus_b_train_c_test.csv
c:\Users\junse\Documents\research\IUBDC 2026\output\time_series_test_a_plus_b_train_c_test.csv
c:\Users\junse\Documents\research\IUBDC 2026\output\time_series_y_train_a_plus_b_train_c_test.csv
c:\Users\junse\Documents\research\IUBDC 2026\output\time_series_y_test_a_plus_b_train_c_test.csv
c:\Users\junse\Documents\research\IUBDC 2026\output\time_series_split_summary_a_plus_b_train_c_test.csv
c:\Users\junse\Documents\research\IUBDC 2026\output\time_series_dropped_rows_summary_a_plus_b_train_c_test.csv
c:\Users\junse\Documents\research\IUBDC 2026\output\time_series_metadata_a_plus_b_train_c_test.json


In [9]:
# ============================================================
# 8. Final sanity checks
# ============================================================

assert time_series_train["split"].eq("train").all()
assert time_series_test["split"].eq("test").all()

assert time_series_train["RecordID"].nunique() == 8000, "Expected A+B train to contain 8000 patients."
assert time_series_test["RecordID"].nunique() == 4000, "Expected C test to contain 4000 patients."

assert len(set(time_series_train["RecordID"].dropna().astype(int)).intersection(
    set(time_series_test["RecordID"].dropna().astype(int))
)) == 0

assert time_series_train["Parameter"].notna().all()
assert time_series_test["Parameter"].notna().all()

if DROP_RECORDID_PARAMETER_ROWS:
    assert not (time_series_train["Parameter"] == "RecordID").any()
    assert not (time_series_test["Parameter"] == "RecordID").any()

display(summary_after)

if LABELS_AVAILABLE:
    label_summary = pd.DataFrame([
        {
            "split": "train_A_plus_B",
            "n_patients": len(time_series_y_train),
            "death_rate": time_series_y_train[TARGET].mean(),
        },
        {
            "split": "test_C",
            "n_patients": len(time_series_y_test),
            "death_rate": time_series_y_test[TARGET].mean(),
        },
    ])
    display(label_summary)


,split,n_rows,n_unique_patients,n_parameters,min_time_minutes,max_time_minutes,missing_parameter_rows,missing_value_numeric_rows
0,train_A_plus_B,3512515,8000,41,0,2880,0,0
1,test_C,1758900,4000,41,0,2880,0,0


,split,n_patients,death_rate
0,train_A_plus_B,8000,0.14025
1,test_C,4000,0.14625


## Notes for later time-series modeling

The exported files are still in **long format**. For models such as MLP, CNN, RNN, or Transformer, you will usually need one extra conversion step.

A common practical option is:

- split 48 hours into fixed bins, for example 12 bins × 4 hours,
- aggregate each variable within each bin using statistics such as mean, min, max, last, and count,
- fit binning/scaling decisions on the training split only,
- use Set C only once as the final held-out test set.

The current notebook only creates the leakage-safe time-series train/test files.
